# vector_checker — разбор расхождений service ↔ 1С

Тетрадка вместо чтения JSON: сводка, таблица cell-mismatch, drill-down по `LOSS_NUMBER`.

**Режимы**
- `USE_SYNTHETIC = True` — демо на синтетике (match / mismatch)
- `False` — реальный `df_for_service` + Excel из `work/excel/`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Ищем examples/querulus независимо от cwd ядра
def _find_querulus_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "integration" / "tests" / "vector_checker").is_dir() and (
            cand / "integration" / "main.py"
        ).is_file():
            return cand
    raise FileNotFoundError("Не найден examples/querulus (нужен integration/main.py)")

QUERULUS_ROOT = _find_querulus_root()
if str(QUERULUS_ROOT) not in sys.path:
    sys.path.insert(0, str(QUERULUS_ROOT))

from integration.tests.vector_checker.compare_features import (
    compare_feature_frames,
    load_excel_export,
    mismatch_feature_counts,
    report_to_frame,
    side_by_side_for_loss,
)
from integration.tests.vector_checker.paths import (
    EXCEL_DIR,
    ensure_work_dirs,
    resolve_df_for_service,
)
from integration.tests.vector_checker.synthetic import write_synthetic_fixtures

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

ensure_work_dirs()
print("QUERULUS_ROOT:", QUERULUS_ROOT)
print("EXCEL_DIR:", EXCEL_DIR)

## 1. Источники

- `USE_SYNTHETIC=True` + `SYNTHETIC_MISMATCH=True` — заранее сломанный Excel (удобно смотреть FAIL)
- для боя: положи выгрузку 1С в `work/excel/` и укажи имя файла

In [ ]:
USE_SYNTHETIC = True
SYNTHETIC_MISMATCH = True  # True → export_1c_mismatch.xlsx; False → export_1c_ok.xlsx

# Боевой режим (USE_SYNTHETIC=False):
EXCEL_NAME = "export.xlsx"  # файл в work/excel/
DF_PATH: str | None = None  # None → автопоиск df_for_service

if USE_SYNTHETIC:
    paths = write_synthetic_fixtures()
    df_path = paths["df"]
    excel_path = paths["excel_bad" if SYNTHETIC_MISMATCH else "excel_ok"]
else:
    df_path = resolve_df_for_service(DF_PATH)
    excel_path = EXCEL_DIR / EXCEL_NAME
    if not excel_path.is_file():
        raise FileNotFoundError(
            f"Нет Excel: {excel_path}. Положи выгрузку 1С в {EXCEL_DIR}"
        )

print("df   :", df_path)
print("excel:", excel_path)

In [ ]:
service = pd.read_parquet(df_path)
excel = load_excel_export(excel_path)

print("service:", service.shape, "excel:", excel.shape)
display(service.head(3))
display(excel.head(3))

## 2. Сводка и клеточные расхождения

`mismatches` — длинная таблица: одна строка = один `LOSS_NUMBER` × одна фича.

In [ ]:
report, mismatches = compare_feature_frames(service, excel)

display(report_to_frame(report))
print("ok:", report.ok)
print("нет в Excel:", report.features_missing_in_excel)
print("нет в service:", report.features_missing_in_service)

display(mismatch_feature_counts(mismatches))
display(mismatches)

## 3. Ключи только в одной стороне

In [ ]:
only_service = pd.DataFrame({"LOSS_NUMBER": report.only_service_losses})
only_excel = pd.DataFrame({"LOSS_NUMBER": report.only_excel_losses})

print("только в service:", len(only_service))
display(only_service.head(20))
print("только в excel:", len(only_excel))
display(only_excel.head(20))

if len(only_service):
    display(service.loc[service["LOSS_NUMBER"].astype(str).isin(only_service["LOSS_NUMBER"])].head())

## 4. Drill-down по одному убытку

Подставь `LOSS_NUMBER` из `mismatches` или `only_service` — увидишь side-by-side по всем общим фичам.

In [ ]:
if not mismatches.empty:
    LOSS_TO_INSPECT = mismatches["LOSS_NUMBER"].iloc[0]
elif report.only_service_losses:
    LOSS_TO_INSPECT = report.only_service_losses[0]
else:
    LOSS_TO_INSPECT = str(service["LOSS_NUMBER"].iloc[0])

print("LOSS_TO_INSPECT:", LOSS_TO_INSPECT)
sbs = side_by_side_for_loss(service, excel, LOSS_TO_INSPECT)
display(sbs.loc[~sbs["match"]] if (~sbs["match"]).any() else sbs)
display(sbs)  # полный вектор рядом

## 5. Фильтр по фиче

Например: все убытки, где разъехался `FILIAL`.

In [ ]:
FEATURE = "FILIAL"  # поменяй на нужную

if mismatches.empty:
    print("Расхождений по ячейкам нет")
else:
    subset = mismatches.loc[mismatches["feature"] == FEATURE]
    display(subset)
    if len(subset):
        keys = subset["LOSS_NUMBER"].astype(str)
        cols = ["LOSS_NUMBER", FEATURE] if FEATURE in service.columns else ["LOSS_NUMBER"]
        left = service.assign(_k=service["LOSS_NUMBER"].astype(str)).loc[
            lambda d: d["_k"].isin(keys), cols
        ].rename(columns={FEATURE: f"{FEATURE}__service"} if FEATURE in cols else {})
        right_cols = [c for c in ["LOSS_NUMBER", FEATURE] if c in excel.columns]
        right = excel.assign(_k=excel["LOSS_NUMBER"].astype(str)).loc[
            lambda d: d["_k"].isin(keys), right_cols
        ].rename(columns={FEATURE: f"{FEATURE}__excel"} if FEATURE in right_cols else {})
        display(
            left.merge(
                right,
                on="LOSS_NUMBER",
                how="outer",
                suffixes=("_svc", "_xl"),
            )
        )